# 06 - Analyses Consolidation for ACM Publication

This notebook consolidates all analyses from the AINarratives study into a single Excel file with multiple sheets.

**Requirements from the paper:**
- Sample characteristics (N=152)
- Expert feedback on dimensions and questionnaires
- MAE and RMSE (synthetic data based on 10 independent runs)
- Correlations: Real vs Synthetic (at individual and questionnaire level)
- Cronbach's α comparison (real vs synthetic)
- Statistical significance tests (t-tests, Cohen's d)
- Population-level vs Individual-level agreement interpretation

**Output:** `publications/AINarratives_Analyses_Consolidated.xlsx`

## 1. Setup & Configuration

In [ ]:
import json
import sys
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from scipy import stats
from sqlalchemy import text
from sqlmodel import Session

# Add src to path
notebook_dir = Path.cwd()
project_root = notebook_dir.parent
sys.path.insert(0, str(project_root / 'src'))

from pain_narratives.core.database import DatabaseManager

# Initialize database
db = DatabaseManager()

# Schema
SCHEMA = 'pain_narratives_acm_202512'

# Output directory
output_dir = project_root / 'publications'
output_dir.mkdir(exist_ok=True)

# Excel output file
EXCEL_OUTPUT = output_dir / f'AINarratives_Analyses_Consolidated_{datetime.now().strftime("%Y%m%d")}.xlsx'

print(f"Output will be saved to: {EXCEL_OUTPUT}")
print(f"Connected to database")

In [ ]:
# Dictionary to store all sheets for Excel export
excel_sheets = {}

## 2. Load All Data

In [ ]:
print("="*80)
print("LOADING DATA FROM DATABASE")
print("="*80)

with Session(db.engine) as session:
    # Real patient data
    df_real_pcs = pd.read_sql(text(f"SELECT * FROM {SCHEMA}.real_patient_pcs ORDER BY participant_id"), session.connection())
    df_real_bpi = pd.read_sql(text(f"SELECT * FROM {SCHEMA}.real_patient_bpi ORDER BY participant_id"), session.connection())
    df_real_tsk = pd.read_sql(text(f"SELECT * FROM {SCHEMA}.real_patient_tsk ORDER BY participant_id"), session.connection())
    
    # Participant demographics (correct table name)
    df_demographics = pd.read_sql(text(f"SELECT * FROM {SCHEMA}.real_patient_demographics ORDER BY participant_id"), session.connection())
    
    # LLM synthetic data (all runs)
    df_llm_pcs = pd.read_sql(text(f"SELECT * FROM {SCHEMA}.llm_pcs_results ORDER BY participant_id, run_number"), session.connection())
    df_llm_bpi = pd.read_sql(text(f"SELECT * FROM {SCHEMA}.llm_bpi_results ORDER BY participant_id, run_number"), session.connection())
    df_llm_tsk = pd.read_sql(text(f"SELECT * FROM {SCHEMA}.llm_tsk_results ORDER BY participant_id, run_number"), session.connection())

n_participants = len(df_demographics)
n_runs = df_llm_pcs['run_number'].nunique()

print(f"\nReal Patient Data:")
print(f"  Demographics: {len(df_demographics)} participants")
print(f"  PCS: {len(df_real_pcs)} records")
print(f"  BPI: {len(df_real_bpi)} records")
print(f"  TSK: {len(df_real_tsk)} records")

print(f"\nLLM Synthetic Data:")
print(f"  Number of runs: {n_runs}")
print(f"  PCS: {len(df_llm_pcs)} records ({len(df_llm_pcs)//n_runs} per run)")
print(f"  BPI: {len(df_llm_bpi)} records ({len(df_llm_bpi)//n_runs} per run)")
print(f"  TSK: {len(df_llm_tsk)} records ({len(df_llm_tsk)//n_runs} per run)")

## 3. Sample Characteristics (Table 1)

In [ ]:
print("="*80)
print("TABLE 1: SAMPLE CHARACTERISTICS")
print("="*80)

# Helper function to parse Spanish year strings to numeric
def parse_years_spanish(value):
    """Convert Spanish year strings like '23 años', '1 año', 'Menos de un año' to numeric."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float)):
        return float(value)
    
    value = str(value).lower().strip()
    
    # Handle "menos de un año" (less than a year)
    if 'menos de un año' in value or 'menos de 1 año' in value:
        return 0.5
    
    # Handle "un año" or "1 año"
    if value in ['un año', '1 año']:
        return 1.0
    
    # Extract number from strings like "23 años", "12 años"
    import re
    match = re.search(r'(\d+)', value)
    if match:
        return float(match.group(1))
    
    return np.nan

# Clean the years columns
if 'years_with_pain' in df_demographics.columns:
    df_demographics['years_with_pain_numeric'] = df_demographics['years_with_pain'].apply(parse_years_spanish)

if 'years_since_diagnosis' in df_demographics.columns:
    df_demographics['years_since_diagnosis_numeric'] = df_demographics['years_since_diagnosis'].apply(parse_years_spanish)

characteristics = []

# Sample Size
characteristics.append({'Characteristic': 'Sample Size', 'Value': str(n_participants)})
characteristics.append({'Characteristic': '', 'Value': ''})

# Demographics
characteristics.append({'Characteristic': 'Demographics', 'Value': ''})

# Age
if 'age' in df_demographics.columns:
    # Convert age to numeric if needed
    df_demographics['age_numeric'] = pd.to_numeric(df_demographics['age'], errors='coerce')
    age_mean = df_demographics['age_numeric'].mean()
    age_sd = df_demographics['age_numeric'].std()
    age_min = df_demographics['age_numeric'].min()
    age_max = df_demographics['age_numeric'].max()
    characteristics.append({'Characteristic': '  Age, years — mean (SD)', 'Value': f'{age_mean:.1f} ({age_sd:.1f})'})
    characteristics.append({'Characteristic': '  Age range', 'Value': f'[{int(age_min)}, {int(age_max)}]'})

# Gender
if 'gender' in df_demographics.columns:
    gender_counts = df_demographics['gender'].value_counts()
    for gender, count in gender_counts.items():
        pct = 100 * count / n_participants
        characteristics.append({'Characteristic': f'  {gender}', 'Value': f'{count} ({pct:.1f}%)'})

# Education Level
if 'education_level' in df_demographics.columns:
    characteristics.append({'Characteristic': '', 'Value': ''})
    characteristics.append({'Characteristic': 'Education Level', 'Value': ''})
    edu_counts = df_demographics['education_level'].value_counts()
    for edu, count in edu_counts.items():
        if pd.notna(edu) and edu != '':
            pct = 100 * count / n_participants
            characteristics.append({'Characteristic': f'  {edu}', 'Value': f'{count} ({pct:.1f}%)'})

# Employment Status
if 'employment_status' in df_demographics.columns:
    characteristics.append({'Characteristic': '', 'Value': ''})
    characteristics.append({'Characteristic': 'Employment Status', 'Value': ''})
    emp_counts = df_demographics['employment_status'].value_counts()
    for emp, count in emp_counts.items():
        if pd.notna(emp) and emp != '':
            pct = 100 * count / n_participants
            characteristics.append({'Characteristic': f'  {emp}', 'Value': f'{count} ({pct:.1f}%)'})

characteristics.append({'Characteristic': '', 'Value': ''})

# Pain Characteristics
characteristics.append({'Characteristic': 'Pain Characteristics', 'Value': ''})

# Primary Pain Diagnosis
if 'pain_cause_primary' in df_demographics.columns:
    diag_counts = df_demographics['pain_cause_primary'].value_counts()
    characteristics.append({'Characteristic': '  Primary Pain Diagnosis', 'Value': ''})
    for diag, count in diag_counts.head(10).items():  # Top 10
        if pd.notna(diag) and diag != '':
            pct = 100 * count / n_participants
            # Truncate long diagnosis names
            diag_display = diag[:40] + '...' if len(str(diag)) > 40 else diag
            characteristics.append({'Characteristic': f'    {diag_display}', 'Value': f'{count} ({pct:.1f}%)'})

if 'years_with_pain_numeric' in df_demographics.columns:
    ywp_mean = df_demographics['years_with_pain_numeric'].mean()
    ywp_sd = df_demographics['years_with_pain_numeric'].std()
    characteristics.append({'Characteristic': '  Years with pain — mean (SD)', 'Value': f'{ywp_mean:.1f} ({ywp_sd:.1f})'})

if 'years_since_diagnosis_numeric' in df_demographics.columns:
    ysd_mean = df_demographics['years_since_diagnosis_numeric'].mean()
    ysd_sd = df_demographics['years_since_diagnosis_numeric'].std()
    characteristics.append({'Characteristic': '  Years since diagnosis — mean (SD)', 'Value': f'{ysd_mean:.1f} ({ysd_sd:.1f})'})

characteristics.append({'Characteristic': '', 'Value': ''})

# Narrative Characteristics  
characteristics.append({'Characteristic': 'Narrative Characteristics', 'Value': ''})
if 'word_count' in df_demographics.columns:
    # Convert to numeric if needed
    df_demographics['word_count_numeric'] = pd.to_numeric(df_demographics['word_count'], errors='coerce')
    wc_mean = df_demographics['word_count_numeric'].mean()
    wc_sd = df_demographics['word_count_numeric'].std()
    wc_min = df_demographics['word_count_numeric'].min()
    wc_max = df_demographics['word_count_numeric'].max()
    characteristics.append({'Characteristic': '  Word count — mean (SD)', 'Value': f'{wc_mean:.0f} ({wc_sd:.0f})'})
    characteristics.append({'Characteristic': '  Word count range', 'Value': f'[{int(wc_min)}, {int(wc_max)}]'})

df_characteristics = pd.DataFrame(characteristics)
excel_sheets['T1_Sample_Characteristics'] = df_characteristics

print(df_characteristics.to_string(index=False))

## 4. Questionnaire Descriptives (Real vs Synthetic)

In [ ]:
print("="*80)
print("TABLE 2: QUESTIONNAIRE DESCRIPTIVES - REAL VS SYNTHETIC")
print("="*80)

# Calculate LLM aggregates (mean across all runs per participant)
df_llm_pcs_agg = df_llm_pcs.groupby('participant_id')['pcs_total'].agg(['mean', 'std']).reset_index()
df_llm_pcs_agg.columns = ['participant_id', 'pcs_total_mean', 'pcs_total_std']

# BPI - use bpi_total_mean if available, otherwise compute
if 'bpi_total_mean' in df_llm_bpi.columns:
    df_llm_bpi_agg = df_llm_bpi.groupby('participant_id')['bpi_total_mean'].agg(['mean', 'std']).reset_index()
    df_llm_bpi_agg.columns = ['participant_id', 'bpi_total_mean', 'bpi_total_std']
else:
    df_llm_bpi['bpi_total_mean_calc'] = df_llm_bpi['bpi_total'] / 10.0
    df_llm_bpi_agg = df_llm_bpi.groupby('participant_id')['bpi_total_mean_calc'].agg(['mean', 'std']).reset_index()
    df_llm_bpi_agg.columns = ['participant_id', 'bpi_total_mean', 'bpi_total_std']

df_llm_tsk_agg = df_llm_tsk.groupby('participant_id')['tsk_total'].agg(['mean', 'std']).reset_index()
df_llm_tsk_agg.columns = ['participant_id', 'tsk_total_mean', 'tsk_total_std']

descriptives = []

# PCS
descriptives.append({
    'Questionnaire': 'PCS Total',
    'N': n_participants,
    'Real_Mean': df_real_pcs['pcs_total'].mean(),
    'Real_SD': df_real_pcs['pcs_total'].std(),
    'Synthetic_Mean': df_llm_pcs_agg['pcs_total_mean'].mean(),
    'Synthetic_SD': df_llm_pcs_agg['pcs_total_mean'].std(),
    'Synthetic_Avg_Within_SD': df_llm_pcs_agg['pcs_total_std'].mean()
})

# BPI Total Mean
descriptives.append({
    'Questionnaire': 'BPI Total (0-10)',
    'N': n_participants,
    'Real_Mean': df_real_bpi['bpi_total_mean'].mean(),
    'Real_SD': df_real_bpi['bpi_total_mean'].std(),
    'Synthetic_Mean': df_llm_bpi_agg['bpi_total_mean'].mean(),
    'Synthetic_SD': df_llm_bpi_agg['bpi_total_mean'].std(),
    'Synthetic_Avg_Within_SD': df_llm_bpi_agg['bpi_total_std'].mean()
})

# BPI Interference
if 'bpi_interference_mean' in df_real_bpi.columns and 'bpi_interference_avg' in df_llm_bpi.columns:
    llm_int_agg = df_llm_bpi.groupby('participant_id')['bpi_interference_avg'].agg(['mean', 'std']).reset_index()
    descriptives.append({
        'Questionnaire': 'BPI Interference',
        'N': n_participants,
        'Real_Mean': df_real_bpi['bpi_interference_mean'].mean(),
        'Real_SD': df_real_bpi['bpi_interference_mean'].std(),
        'Synthetic_Mean': llm_int_agg['mean'].mean(),
        'Synthetic_SD': llm_int_agg['mean'].std(),
        'Synthetic_Avg_Within_SD': llm_int_agg['std'].mean()
    })

# BPI Intensity
if 'bpi_intensity_mean' in df_real_bpi.columns and 'bpi_intensity_avg' in df_llm_bpi.columns:
    llm_inten_agg = df_llm_bpi.groupby('participant_id')['bpi_intensity_avg'].agg(['mean', 'std']).reset_index()
    descriptives.append({
        'Questionnaire': 'BPI Intensity',
        'N': n_participants,
        'Real_Mean': df_real_bpi['bpi_intensity_mean'].mean(),
        'Real_SD': df_real_bpi['bpi_intensity_mean'].std(),
        'Synthetic_Mean': llm_inten_agg['mean'].mean(),
        'Synthetic_SD': llm_inten_agg['mean'].std(),
        'Synthetic_Avg_Within_SD': llm_inten_agg['std'].mean()
    })

# TSK
descriptives.append({
    'Questionnaire': 'TSK Total',
    'N': n_participants,
    'Real_Mean': df_real_tsk['tsk_total'].mean(),
    'Real_SD': df_real_tsk['tsk_total'].std(),
    'Synthetic_Mean': df_llm_tsk_agg['tsk_total_mean'].mean(),
    'Synthetic_SD': df_llm_tsk_agg['tsk_total_mean'].std(),
    'Synthetic_Avg_Within_SD': df_llm_tsk_agg['tsk_total_std'].mean()
})

df_descriptives = pd.DataFrame(descriptives)
df_descriptives = df_descriptives.round(2)
excel_sheets['T2_Questionnaire_Descriptives'] = df_descriptives

print(f"\nBased on {n_runs} independent runs")
print(df_descriptives.to_string(index=False))

## 5. MAE and RMSE: Real vs Synthetic

In [ ]:
print("="*80)
print("TABLE 3: MAE AND RMSE - REAL VS SYNTHETIC")
print("="*80)

def compute_error_metrics(df, real_col, synth_col):
    """Compute MAE, RMSE, and correlation between real and synthetic."""
    valid = df[[real_col, synth_col]].dropna()
    real = valid[real_col].values
    synth = valid[synth_col].values
    
    mae = np.mean(np.abs(real - synth))
    rmse = np.sqrt(np.mean((real - synth) ** 2))
    
    # Pearson correlation
    if len(valid) > 2:
        r, p = stats.pearsonr(real, synth)
    else:
        r, p = np.nan, np.nan
    
    return {'n': len(valid), 'mae': mae, 'rmse': rmse, 'r': r, 'p_value': p}

# Merge real and synthetic
df_pcs_merged = df_real_pcs[['participant_id', 'pcs_total']].merge(
    df_llm_pcs_agg, on='participant_id'
)

# For BPI, rename columns explicitly to avoid confusion
df_bpi_real_subset = df_real_bpi[['participant_id', 'bpi_total_mean', 'bpi_interference_mean', 'bpi_intensity_mean']].copy()
df_bpi_real_subset = df_bpi_real_subset.rename(columns={
    'bpi_total_mean': 'bpi_total_mean_real',
    'bpi_interference_mean': 'bpi_interference_real',
    'bpi_intensity_mean': 'bpi_intensity_real'
})

df_bpi_synth_subset = df_llm_bpi_agg[['participant_id', 'bpi_total_mean']].copy()
df_bpi_synth_subset = df_bpi_synth_subset.rename(columns={
    'bpi_total_mean': 'bpi_total_mean_synth'
})

df_bpi_merged = df_bpi_real_subset.merge(df_bpi_synth_subset, on='participant_id')

df_tsk_merged = df_real_tsk[['participant_id', 'tsk_total']].merge(
    df_llm_tsk_agg, on='participant_id'
)

# Also merge interference and intensity
if 'bpi_interference_avg' in df_llm_bpi.columns:
    llm_int_agg = df_llm_bpi.groupby('participant_id')['bpi_interference_avg'].mean().reset_index()
    llm_int_agg.columns = ['participant_id', 'bpi_interference_synth']
    df_bpi_merged = df_bpi_merged.merge(llm_int_agg, on='participant_id', how='left')

if 'bpi_intensity_avg' in df_llm_bpi.columns:
    llm_inten_agg = df_llm_bpi.groupby('participant_id')['bpi_intensity_avg'].mean().reset_index()
    llm_inten_agg.columns = ['participant_id', 'bpi_intensity_synth']
    df_bpi_merged = df_bpi_merged.merge(llm_inten_agg, on='participant_id', how='left')

error_results = []

# PCS
metrics = compute_error_metrics(df_pcs_merged, 'pcs_total', 'pcs_total_mean')
metrics['Questionnaire'] = 'PCS Total'
error_results.append(metrics)

# BPI Total
metrics = compute_error_metrics(df_bpi_merged, 'bpi_total_mean_real', 'bpi_total_mean_synth')
metrics['Questionnaire'] = 'BPI Total (0-10)'
error_results.append(metrics)

# BPI Interference
if 'bpi_interference_synth' in df_bpi_merged.columns:
    metrics = compute_error_metrics(df_bpi_merged, 'bpi_interference_real', 'bpi_interference_synth')
    metrics['Questionnaire'] = 'BPI Interference'
    error_results.append(metrics)

# BPI Intensity
if 'bpi_intensity_synth' in df_bpi_merged.columns:
    metrics = compute_error_metrics(df_bpi_merged, 'bpi_intensity_real', 'bpi_intensity_synth')
    metrics['Questionnaire'] = 'BPI Intensity'
    error_results.append(metrics)

# TSK
metrics = compute_error_metrics(df_tsk_merged, 'tsk_total', 'tsk_total_mean')
metrics['Questionnaire'] = 'TSK Total'
error_results.append(metrics)

df_errors = pd.DataFrame(error_results)
df_errors = df_errors[['Questionnaire', 'n', 'mae', 'rmse', 'r', 'p_value']]
df_errors = df_errors.round(3)
excel_sheets['T3_MAE_RMSE'] = df_errors

print(f"\nBased on {n_runs} independent runs (synthetic = mean across runs)")
print(df_errors.to_string(index=False))

## 6. Statistical Significance Tests

In [ ]:
print("="*80)
print("TABLE 4: PAIRED T-TESTS - REAL VS SYNTHETIC")
print("="*80)

def cohen_d(x, y):
    """Calculate Cohen's d effect size."""
    nx, ny = len(x), len(y)
    pooled_std = np.sqrt(((nx-1)*np.std(x, ddof=1)**2 + (ny-1)*np.std(y, ddof=1)**2) / (nx+ny-2))
    return (np.mean(x) - np.mean(y)) / pooled_std if pooled_std > 0 else 0

def interpret_cohens_d(d):
    """Interpret Cohen's d effect size."""
    d = abs(d)
    if d >= 0.8: return "Large"
    elif d >= 0.5: return "Medium"
    elif d >= 0.2: return "Small"
    else: return "Negligible"

ttest_results = []

# PCS
real = df_pcs_merged['pcs_total'].values
synth = df_pcs_merged['pcs_total_mean'].values
t_stat, p_val = stats.ttest_rel(real, synth)
d = cohen_d(real, synth)
ttest_results.append({
    'Questionnaire': 'PCS Total',
    'Real_Mean': np.mean(real),
    'Real_SD': np.std(real, ddof=1),
    'Synthetic_Mean': np.mean(synth),
    'Synthetic_SD': np.std(synth, ddof=1),
    'Mean_Diff': np.mean(real) - np.mean(synth),
    't_statistic': t_stat,
    'p_value': p_val,
    'Cohens_d': d,
    'Effect_Size': interpret_cohens_d(d),
    'Significant': 'Yes' if p_val < 0.05 else 'No'
})

# BPI Total
real = df_bpi_merged['bpi_total_mean_real'].values
synth = df_bpi_merged['bpi_total_mean_synth'].values
t_stat, p_val = stats.ttest_rel(real, synth)
d = cohen_d(real, synth)
ttest_results.append({
    'Questionnaire': 'BPI Total (0-10)',
    'Real_Mean': np.mean(real),
    'Real_SD': np.std(real, ddof=1),
    'Synthetic_Mean': np.mean(synth),
    'Synthetic_SD': np.std(synth, ddof=1),
    'Mean_Diff': np.mean(real) - np.mean(synth),
    't_statistic': t_stat,
    'p_value': p_val,
    'Cohens_d': d,
    'Effect_Size': interpret_cohens_d(d),
    'Significant': 'Yes' if p_val < 0.05 else 'No'
})

# BPI Interference
if 'bpi_interference_synth' in df_bpi_merged.columns:
    real = df_bpi_merged['bpi_interference_real'].values
    synth = df_bpi_merged['bpi_interference_synth'].values
    t_stat, p_val = stats.ttest_rel(real, synth)
    d = cohen_d(real, synth)
    ttest_results.append({
        'Questionnaire': 'BPI Interference',
        'Real_Mean': np.mean(real),
        'Real_SD': np.std(real, ddof=1),
        'Synthetic_Mean': np.mean(synth),
        'Synthetic_SD': np.std(synth, ddof=1),
        'Mean_Diff': np.mean(real) - np.mean(synth),
        't_statistic': t_stat,
        'p_value': p_val,
        'Cohens_d': d,
        'Effect_Size': interpret_cohens_d(d),
        'Significant': 'Yes' if p_val < 0.05 else 'No'
    })

# BPI Intensity
if 'bpi_intensity_synth' in df_bpi_merged.columns:
    real = df_bpi_merged['bpi_intensity_real'].values
    synth = df_bpi_merged['bpi_intensity_synth'].values
    t_stat, p_val = stats.ttest_rel(real, synth)
    d = cohen_d(real, synth)
    ttest_results.append({
        'Questionnaire': 'BPI Intensity',
        'Real_Mean': np.mean(real),
        'Real_SD': np.std(real, ddof=1),
        'Synthetic_Mean': np.mean(synth),
        'Synthetic_SD': np.std(synth, ddof=1),
        'Mean_Diff': np.mean(real) - np.mean(synth),
        't_statistic': t_stat,
        'p_value': p_val,
        'Cohens_d': d,
        'Effect_Size': interpret_cohens_d(d),
        'Significant': 'Yes' if p_val < 0.05 else 'No'
    })

# TSK
real = df_tsk_merged['tsk_total'].values
synth = df_tsk_merged['tsk_total_mean'].values
t_stat, p_val = stats.ttest_rel(real, synth)
d = cohen_d(real, synth)
ttest_results.append({
    'Questionnaire': 'TSK Total',
    'Real_Mean': np.mean(real),
    'Real_SD': np.std(real, ddof=1),
    'Synthetic_Mean': np.mean(synth),
    'Synthetic_SD': np.std(synth, ddof=1),
    'Mean_Diff': np.mean(real) - np.mean(synth),
    't_statistic': t_stat,
    'p_value': p_val,
    'Cohens_d': d,
    'Effect_Size': interpret_cohens_d(d),
    'Significant': 'Yes' if p_val < 0.05 else 'No'
})

df_ttests = pd.DataFrame(ttest_results)
df_ttests = df_ttests.round(3)
excel_sheets['T4_Paired_Ttests'] = df_ttests

print(df_ttests.to_string(index=False))

## 7. Cronbach's Alpha Comparison

In [ ]:
print("="*80)
print("TABLE 5: CRONBACH'S ALPHA - REAL VS SYNTHETIC")
print("="*80)

def cronbachs_alpha(df_items):
    """Calculate Cronbach's alpha for a set of item columns."""
    df_items = df_items.apply(pd.to_numeric, errors='coerce').dropna()
    if len(df_items) < 2 or len(df_items.columns) < 2:
        return np.nan
    
    item_variances = df_items.var(axis=0, ddof=1)
    total_var = df_items.sum(axis=1).var(ddof=1)
    n_items = len(df_items.columns)
    
    if total_var == 0:
        return np.nan
    
    alpha = (n_items / (n_items - 1)) * (1 - item_variances.sum() / total_var)
    return alpha

alpha_results = []

# PCS Items
pcs_items = [f'pcs_{i:02d}' for i in range(1, 14)]
pcs_real_items = [c for c in pcs_items if c in df_real_pcs.columns]
pcs_llm_items = [c for c in pcs_items if c in df_llm_pcs.columns]

if pcs_real_items:
    alpha_real_pcs = cronbachs_alpha(df_real_pcs[pcs_real_items])
else:
    alpha_real_pcs = np.nan

# LLM PCS - calculate alpha per run, then average
if pcs_llm_items:
    alphas_llm_pcs = []
    for run in df_llm_pcs['run_number'].unique():
        run_data = df_llm_pcs[df_llm_pcs['run_number'] == run][pcs_llm_items]
        alpha = cronbachs_alpha(run_data)
        if not np.isnan(alpha):
            alphas_llm_pcs.append(alpha)
    alpha_llm_pcs_mean = np.mean(alphas_llm_pcs) if alphas_llm_pcs else np.nan
    alpha_llm_pcs_sd = np.std(alphas_llm_pcs, ddof=1) if len(alphas_llm_pcs) > 1 else 0
else:
    alpha_llm_pcs_mean, alpha_llm_pcs_sd = np.nan, np.nan

alpha_results.append({
    'Questionnaire': 'PCS (13 items)',
    'Real_Alpha': alpha_real_pcs,
    'Synthetic_Alpha_Mean': alpha_llm_pcs_mean,
    'Synthetic_Alpha_SD': alpha_llm_pcs_sd,
    'N_Runs': n_runs
})

# BPI Items
bpi_items = ['bpiq11', 'bpiq12', 'bpiq13', 'bpiq15', 'bpiq16', 'bpiq17', 'bpiq28', 'bpiq39', 'bpiq410', 'bpiq511']
bpi_real_items = [c for c in bpi_items if c in df_real_bpi.columns]
bpi_llm_items = [c for c in bpi_items if c in df_llm_bpi.columns]

if bpi_real_items:
    alpha_real_bpi = cronbachs_alpha(df_real_bpi[bpi_real_items])
else:
    alpha_real_bpi = np.nan

if bpi_llm_items:
    alphas_llm_bpi = []
    for run in df_llm_bpi['run_number'].unique():
        run_data = df_llm_bpi[df_llm_bpi['run_number'] == run][bpi_llm_items]
        alpha = cronbachs_alpha(run_data)
        if not np.isnan(alpha):
            alphas_llm_bpi.append(alpha)
    alpha_llm_bpi_mean = np.mean(alphas_llm_bpi) if alphas_llm_bpi else np.nan
    alpha_llm_bpi_sd = np.std(alphas_llm_bpi, ddof=1) if len(alphas_llm_bpi) > 1 else 0
else:
    alpha_llm_bpi_mean, alpha_llm_bpi_sd = np.nan, np.nan

alpha_results.append({
    'Questionnaire': 'BPI (10 items)',
    'Real_Alpha': alpha_real_bpi,
    'Synthetic_Alpha_Mean': alpha_llm_bpi_mean,
    'Synthetic_Alpha_SD': alpha_llm_bpi_sd,
    'N_Runs': n_runs
})

# TSK Items
tsk_items = [f'tsk_{i:02d}' for i in range(1, 12)]
tsk_real_items = [c for c in tsk_items if c in df_real_tsk.columns]
tsk_llm_items = [c for c in tsk_items if c in df_llm_tsk.columns]

if tsk_real_items:
    alpha_real_tsk = cronbachs_alpha(df_real_tsk[tsk_real_items])
else:
    alpha_real_tsk = np.nan

if tsk_llm_items:
    alphas_llm_tsk = []
    for run in df_llm_tsk['run_number'].unique():
        run_data = df_llm_tsk[df_llm_tsk['run_number'] == run][tsk_llm_items]
        alpha = cronbachs_alpha(run_data)
        if not np.isnan(alpha):
            alphas_llm_tsk.append(alpha)
    alpha_llm_tsk_mean = np.mean(alphas_llm_tsk) if alphas_llm_tsk else np.nan
    alpha_llm_tsk_sd = np.std(alphas_llm_tsk, ddof=1) if len(alphas_llm_tsk) > 1 else 0
else:
    alpha_llm_tsk_mean, alpha_llm_tsk_sd = np.nan, np.nan

alpha_results.append({
    'Questionnaire': 'TSK (11 items)',
    'Real_Alpha': alpha_real_tsk,
    'Synthetic_Alpha_Mean': alpha_llm_tsk_mean,
    'Synthetic_Alpha_SD': alpha_llm_tsk_sd,
    'N_Runs': n_runs
})

df_alphas = pd.DataFrame(alpha_results)
df_alphas = df_alphas.round(3)
excel_sheets['T5_Cronbachs_Alpha'] = df_alphas

print(f"\nSynthetic alpha calculated per run, then averaged across {n_runs} runs")
print(df_alphas.to_string(index=False))

## 8. Questionnaire Inter-Correlations

In [ ]:
print("="*80)
print("TABLE 6: INTER-QUESTIONNAIRE CORRELATIONS")
print("="*80)

# Build correlation matrices for Real and Synthetic

# Real correlations
df_real_merged = df_real_pcs[['participant_id', 'pcs_total']].merge(
    df_real_bpi[['participant_id', 'bpi_total_mean']], on='participant_id'
).merge(
    df_real_tsk[['participant_id', 'tsk_total']], on='participant_id'
)

real_corr = df_real_merged[['pcs_total', 'bpi_total_mean', 'tsk_total']].corr(method='spearman')
real_corr.columns = ['PCS', 'BPI', 'TSK']
real_corr.index = ['PCS', 'BPI', 'TSK']

# Synthetic correlations (using mean across runs)
df_synth_merged = df_llm_pcs_agg[['participant_id', 'pcs_total_mean']].merge(
    df_llm_bpi_agg[['participant_id', 'bpi_total_mean']], on='participant_id'
).merge(
    df_llm_tsk_agg[['participant_id', 'tsk_total_mean']], on='participant_id'
)

synth_corr = df_synth_merged[['pcs_total_mean', 'bpi_total_mean', 'tsk_total_mean']].corr(method='spearman')
synth_corr.columns = ['PCS', 'BPI', 'TSK']
synth_corr.index = ['PCS', 'BPI', 'TSK']

print("\nReal Sample Correlations (Spearman):")
print(real_corr.round(3).to_string())

print("\nSynthetic Sample Correlations (Spearman):")
print(synth_corr.round(3).to_string())

# Create comparison table
corr_comparison = []
pairs = [('PCS', 'BPI'), ('PCS', 'TSK'), ('BPI', 'TSK')]
for q1, q2 in pairs:
    corr_comparison.append({
        'Comparison': f'{q1} vs {q2}',
        'Real_r': real_corr.loc[q1, q2],
        'Synthetic_r': synth_corr.loc[q1, q2],
        'Difference': real_corr.loc[q1, q2] - synth_corr.loc[q1, q2]
    })

df_corr_comparison = pd.DataFrame(corr_comparison)
df_corr_comparison = df_corr_comparison.round(3)
excel_sheets['T6_InterQ_Correlations'] = df_corr_comparison

print("\nCorrelation Comparison:")
print(df_corr_comparison.to_string(index=False))

## 9. LLM Consistency Across Runs

In [ ]:
print("="*80)
print("TABLE 7: LLM CONSISTENCY - SD ACROSS RUNS")
print("="*80)

consistency_results = []

# PCS
pcs_std_per_participant = df_llm_pcs.groupby('participant_id')['pcs_total'].std()
consistency_results.append({
    'Questionnaire': 'PCS Total',
    'Mean_SD_Across_Runs': pcs_std_per_participant.mean(),
    'Min_SD': pcs_std_per_participant.min(),
    'Max_SD': pcs_std_per_participant.max(),
    'N_Runs': n_runs
})

# BPI
if 'bpi_total_mean' in df_llm_bpi.columns:
    bpi_std_per_participant = df_llm_bpi.groupby('participant_id')['bpi_total_mean'].std()
else:
    bpi_std_per_participant = (df_llm_bpi.groupby('participant_id')['bpi_total'].std()) / 10

consistency_results.append({
    'Questionnaire': 'BPI Total (0-10)',
    'Mean_SD_Across_Runs': bpi_std_per_participant.mean(),
    'Min_SD': bpi_std_per_participant.min(),
    'Max_SD': bpi_std_per_participant.max(),
    'N_Runs': n_runs
})

# TSK
tsk_std_per_participant = df_llm_tsk.groupby('participant_id')['tsk_total'].std()
consistency_results.append({
    'Questionnaire': 'TSK Total',
    'Mean_SD_Across_Runs': tsk_std_per_participant.mean(),
    'Min_SD': tsk_std_per_participant.min(),
    'Max_SD': tsk_std_per_participant.max(),
    'N_Runs': n_runs
})

df_consistency = pd.DataFrame(consistency_results)
df_consistency = df_consistency.round(3)
excel_sheets['T7_LLM_Consistency'] = df_consistency

print(f"\nLower SD indicates more consistent LLM responses across {n_runs} runs")
print(df_consistency.to_string(index=False))

## 10. Population vs Individual Agreement Summary

In [ ]:
print("="*80)
print("TABLE 8: POPULATION VS INDIVIDUAL AGREEMENT SUMMARY")
print("="*80)

def interpret_r(r):
    r = abs(r)
    if r >= 0.7: return "Strong"
    elif r >= 0.5: return "Moderate"
    elif r >= 0.3: return "Weak"
    else: return "Very Weak"

agreement_summary = []

for ttest_row in ttest_results:
    q = ttest_row['Questionnaire']
    
    # Find corresponding error metrics
    error_row = next((e for e in error_results if e['Questionnaire'] == q), {})
    
    agreement_summary.append({
        'Questionnaire': q,
        # Population level
        'Real_Mean_SD': f"{ttest_row['Real_Mean']:.2f} ({ttest_row['Real_SD']:.2f})",
        'Synth_Mean_SD': f"{ttest_row['Synthetic_Mean']:.2f} ({ttest_row['Synthetic_SD']:.2f})",
        'Mean_Diff': ttest_row['Mean_Diff'],
        't_stat': ttest_row['t_statistic'],
        'p_value': ttest_row['p_value'],
        'Cohens_d': ttest_row['Cohens_d'],
        'Pop_Similar': 'Yes' if ttest_row['p_value'] >= 0.05 else 'No',
        # Individual level
        'MAE': error_row.get('mae', np.nan),
        'RMSE': error_row.get('rmse', np.nan),
        'Pearson_r': error_row.get('r', np.nan),
        'r_Strength': interpret_r(error_row.get('r', 0))
    })

df_agreement = pd.DataFrame(agreement_summary)
df_agreement = df_agreement.round(3)
excel_sheets['T8_Agreement_Summary'] = df_agreement

print("\nPop_Similar: 'Yes' if p >= 0.05 (populations statistically similar)")
print("r_Strength: Interpretation of individual-level correlation\n")
print(df_agreement.to_string(index=False))

## 11. Per-Run Statistics (for reporting mean ± SD across 10 runs)

In [ ]:
print("="*80)
print("TABLE 9: PER-RUN STATISTICS")
print("="*80)

per_run_stats = []

for run in sorted(df_llm_pcs['run_number'].unique()):
    pcs_run = df_llm_pcs[df_llm_pcs['run_number'] == run]['pcs_total']
    
    if 'bpi_total_mean' in df_llm_bpi.columns:
        bpi_run = df_llm_bpi[df_llm_bpi['run_number'] == run]['bpi_total_mean']
    else:
        bpi_run = df_llm_bpi[df_llm_bpi['run_number'] == run]['bpi_total'] / 10
    
    tsk_run = df_llm_tsk[df_llm_tsk['run_number'] == run]['tsk_total']
    
    per_run_stats.append({
        'Run': run,
        'PCS_Mean': pcs_run.mean(),
        'PCS_SD': pcs_run.std(),
        'BPI_Mean': bpi_run.mean(),
        'BPI_SD': bpi_run.std(),
        'TSK_Mean': tsk_run.mean(),
        'TSK_SD': tsk_run.std(),
        'N': len(pcs_run)
    })

df_per_run = pd.DataFrame(per_run_stats)
df_per_run = df_per_run.round(2)

# Add summary row
summary_row = {
    'Run': 'Mean (SD)',
    'PCS_Mean': f"{df_per_run['PCS_Mean'].mean():.2f} ({df_per_run['PCS_Mean'].std():.2f})",
    'PCS_SD': f"{df_per_run['PCS_SD'].mean():.2f}",
    'BPI_Mean': f"{df_per_run['BPI_Mean'].mean():.2f} ({df_per_run['BPI_Mean'].std():.2f})",
    'BPI_SD': f"{df_per_run['BPI_SD'].mean():.2f}",
    'TSK_Mean': f"{df_per_run['TSK_Mean'].mean():.2f} ({df_per_run['TSK_Mean'].std():.2f})",
    'TSK_SD': f"{df_per_run['TSK_SD'].mean():.2f}",
    'N': n_participants
}

excel_sheets['T9_Per_Run_Stats'] = df_per_run

print(df_per_run.to_string(index=False))
print("\n" + "-"*80)
print("Summary across runs:")
print(f"  PCS: Mean = {df_per_run['PCS_Mean'].mean():.2f} (SD = {df_per_run['PCS_Mean'].std():.2f})")
print(f"  BPI: Mean = {df_per_run['BPI_Mean'].mean():.2f} (SD = {df_per_run['BPI_Mean'].std():.2f})")
print(f"  TSK: Mean = {df_per_run['TSK_Mean'].mean():.2f} (SD = {df_per_run['TSK_Mean'].std():.2f})")

## 12. Interpretation Guide

In [ ]:
interpretation_text = """
INTERPRETATION GUIDE FOR ACM PAPER
===================================

1. POPULATION-LEVEL VALIDITY (Tables 2, 4)
   - Non-significant p-values (p >= 0.05) indicate groups are statistically similar
   - Small Cohen's d (< 0.2) indicates negligible practical difference
   - If both conditions met: LLM generates data with similar DISTRIBUTIONAL PROPERTIES

2. INDIVIDUAL-LEVEL ACCURACY (Table 3)
   - MAE/RMSE: Average error in predicting individual scores
   - Pearson r: How well individual rank-ordering is preserved
   - High MAE + Moderate r: LLM captures patterns but not exact values

3. THE KEY DISTINCTION
   ┌─────────────────────────────────────────────────────────────────────┐
   │  DISTRIBUTIONAL VALIDITY  ≠  INDIVIDUAL PREDICTION ACCURACY        │
   │                                                                     │
   │  The LLM can generate a POPULATION of synthetic patients that      │
   │  statistically resembles real chronic pain patients, WITHOUT       │
   │  being able to precisely predict what SPECIFIC individuals scored. │
   └─────────────────────────────────────────────────────────────────────┘

4. INTERNAL CONSISTENCY (Table 5)
   - Cronbach's α > 0.7 indicates acceptable internal consistency
   - Similar α values between real and synthetic suggest LLM preserves
     the questionnaire's psychometric structure

5. CORRELATION STRUCTURE (Table 6)
   - Similar inter-questionnaire correlations suggest LLM preserves
     the relationships between pain constructs

6. LLM CONSISTENCY (Table 7)
   - Low SD across runs indicates reproducible/stable LLM outputs
   - This supports the reliability of synthetic data generation

SUGGESTED PAPER TEXT:
---------------------
"We evaluated the validity of LLM-generated questionnaire responses at two levels:
population-level (distributional similarity) and individual-level (prediction accuracy).

POPULATION-LEVEL VALIDITY:
Paired t-tests comparing real patient scores with LLM-generated synthetic scores
(averaged across {n_runs} independent runs) revealed X/Y questionnaire measures showed
no statistically significant differences (p > 0.05). Effect sizes (Cohen's d) were
uniformly small to negligible, indicating the synthetic data preserves the distributional
properties of real patient responses.

INDIVIDUAL-LEVEL ACCURACY:
While population distributions were well-matched, individual-level prediction showed
moderate accuracy. Mean Absolute Errors ranged from X.XX to Y.YY points, with Pearson
correlations between real and synthetic scores ranging from r = 0.XX to r = 0.YY.
This indicates the LLM preserves relative rank-ordering of patients while showing
expected variation in absolute score prediction.

INTERPRETATION:
These findings suggest LLMs can generate synthetic patient populations that are
statistically representative of real chronic pain patients for research purposes.
However, the moderate individual-level accuracy indicates that narrative text alone
does not contain sufficient information to precisely predict specific questionnaire
responses."
""".format(n_runs=n_runs)

print(interpretation_text)

# Save as text in Excel
df_interpretation = pd.DataFrame({'Interpretation_Guide': interpretation_text.split('\n')})
excel_sheets['Z_Interpretation_Guide'] = df_interpretation

## 13. System Usability Scale (SUS) Results

SUS measures perceived system usability from expert users. Scores range 0-100.

**Benchmarks:** <50 Poor, 50-62 Marginal, 62-68 Average, 68-80 Good, 80-90 Excellent, >90 Best

In [ ]:
print("="*80)
print("TABLE 10: SUS USABILITY RESULTS")
print("="*80)

# Load SUS data from database
sus_query = f"SELECT * FROM {SCHEMA}.sus_usability_results"
try:
    df_sus = pd.read_sql(sus_query, db.engine)
    print(f"Loaded {len(df_sus)} SUS responses from database")
except Exception as e:
    # Fallback to CSV
    print(f"Database query failed, loading from CSV: {e}")
    df_sus = pd.read_csv('../data/sus_responses.csv')

# SUS item descriptions
sus_item_descriptions = {
    'q1': 'I would like to use this system frequently',
    'q2': 'System unnecessarily complex (R)',
    'q3': 'System was easy to use',
    'q4': 'Need support to use system (R)',
    'q5': 'Functions well integrated',
    'q6': 'Too much inconsistency (R)',
    'q7': 'Most people would learn quickly',
    'q8': 'System very cumbersome (R)',
    'q9': 'Felt confident using system',
    'q10': 'Need to learn a lot before use (R)'
}

# Calculate item-level statistics
sus_items = ['q1', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10']
item_stats = []
for item in sus_items:
    item_stats.append({
        'Item': item.upper(),
        'Description': sus_item_descriptions[item],
        'Mean': df_sus[item].mean(),
        'SD': df_sus[item].std()
    })

df_sus_items = pd.DataFrame(item_stats)

# SUS score summary
mean_sus = df_sus['sus_score'].mean()
std_sus = df_sus['sus_score'].std()

# Determine grade
if mean_sus >= 90: grade, rating = 'A+', 'Best Imaginable'
elif mean_sus >= 80: grade, rating = 'A', 'Excellent'
elif mean_sus >= 68: grade, rating = 'B', 'Good'
elif mean_sus >= 62: grade, rating = 'C', 'Average'
elif mean_sus >= 50: grade, rating = 'D', 'Marginal'
else: grade, rating = 'F', 'Poor'

# Cronbach's alpha
item_data = df_sus[sus_items].values
n_items = len(sus_items)
item_variances = np.var(item_data, axis=0, ddof=1)
total_variance = np.var(np.sum(item_data, axis=1), ddof=1)
alpha_sus = (n_items / (n_items - 1)) * (1 - np.sum(item_variances) / total_variance)

print(f"\n📊 SUS Item Statistics:")
print(df_sus_items.to_string(index=False))

print(f"\n📊 SUS Summary:")
print(f"   N Respondents: {len(df_sus)}")
print(f"   Mean SUS Score: {mean_sus:.2f} (SD: {std_sus:.2f})")
print(f"   Range: [{df_sus['sus_score'].min():.1f}, {df_sus['sus_score'].max():.1f}]")
print(f"   Grade: {grade} ({rating})")
print(f"   Cronbach's Alpha: {alpha_sus:.3f}")

# Create summary table
sus_summary = pd.DataFrame([
    {'Metric': 'N Respondents', 'Value': str(len(df_sus))},
    {'Metric': 'Mean SUS Score (SD)', 'Value': f'{mean_sus:.2f} ({std_sus:.2f})'},
    {'Metric': 'Median', 'Value': f'{df_sus["sus_score"].median():.2f}'},
    {'Metric': 'Range', 'Value': f'[{df_sus["sus_score"].min():.1f}, {df_sus["sus_score"].max():.1f}]'},
    {'Metric': 'Grade', 'Value': f'{grade} ({rating})'},
    {'Metric': 'Cronbach Alpha', 'Value': f'{alpha_sus:.3f}'}
])

# Add to excel sheets
excel_sheets['T10_SUS_Items'] = df_sus_items
excel_sheets['T10_SUS_Summary'] = sus_summary
print(f"\n✅ Added SUS tables to Excel export")

## 14. Export to Excel

In [ ]:
print("="*80)
print("EXPORTING TO EXCEL")
print("="*80)

# Create Excel writer
with pd.ExcelWriter(EXCEL_OUTPUT, engine='openpyxl') as writer:
    for sheet_name, df in excel_sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)
        print(f"  Exported: {sheet_name} ({len(df)} rows)")

print(f"\n" + "="*80)
print(f"Excel file saved to: {EXCEL_OUTPUT}")
print("="*80)

## 15. Summary Statistics for Paper

In [ ]:
print("="*80)
print("SUMMARY STATISTICS FOR PAPER")
print("="*80)

print(f"""
METHODS SECTION:
- Sample size: N = {n_participants}
- Number of independent experimental runs: {n_runs}
- Model: GPT-5 (temperature = 0.0)

RESULTS SECTION:

Questionnaire Descriptives (Real vs Synthetic):
""")

for _, row in df_descriptives.iterrows():
    print(f"  {row['Questionnaire']}:")
    print(f"    Real: M = {row['Real_Mean']:.2f}, SD = {row['Real_SD']:.2f}")
    print(f"    Synthetic: M = {row['Synthetic_Mean']:.2f}, SD = {row['Synthetic_SD']:.2f}")

print("\nStatistical Tests:")
n_similar = sum(1 for r in ttest_results if r['p_value'] >= 0.05)
print(f"  {n_similar}/{len(ttest_results)} questionnaire measures showed no significant difference (p >= 0.05)")

print("\nError Metrics:")
mae_range = f"{min(r['mae'] for r in error_results):.2f} - {max(r['mae'] for r in error_results):.2f}"
r_range = f"{min(r['r'] for r in error_results):.3f} - {max(r['r'] for r in error_results):.3f}"
print(f"  MAE range: {mae_range}")
print(f"  Pearson r range: {r_range}")

print("\nCronbach's Alpha:")
for _, row in df_alphas.iterrows():
    print(f"  {row['Questionnaire']}: Real α = {row['Real_Alpha']:.2f}, Synthetic α = {row['Synthetic_Alpha_Mean']:.2f} (SD = {row['Synthetic_Alpha_SD']:.3f})")

## 16. Notebook Complete

In [ ]:
print("\n" + "="*80)
print("NOTEBOOK 06 - ANALYSES CONSOLIDATION COMPLETE")
print("="*80)
print(f"\nOutput file: {EXCEL_OUTPUT}")
print(f"\nSheets exported:")
for sheet_name in excel_sheets.keys():
    print(f"  - {sheet_name}")
print("\nRe-run this notebook after completing all 10 runs to generate final publication tables.")